# 08 - Dates and Timestamps

**Suggested time: 15 minutes**

Dates often arrive as text. Convert them before filtering, grouping, calculating, or displaying them.

## Learning objectives

By the end of this notebook, you will be able to:

- parse dates and timestamps with explicit formats;
- detect invalid temporal values;
- derive calendar fields and calculate differences; and
- convert UTC timestamps to a named local time zone.

## Prerequisite recap

Notebook 03 introduced `DATE` and `TIMESTAMP`. Use safe parsing when a source can contain malformed values, and keep temporal columns typed until a final display label is required.

In [ ]:
from pyspark.sql import functions as F

deliveries_raw = spark.createDataFrame(
    [
        ('O100', '2026-01-15', '2026-01-18', '2026-01-15 08:30:00'),
        ('O101', '2026-02-01', '2026-02-06', '2026-02-01 14:45:00'),
        ('O102', 'not a date', '2026-03-14', 'bad timestamp'),
    ],
    ['order_id', 'order_date_text', 'promised_date_text', 'event_time_utc_text'],
)
deliveries_raw.show(truncate=False)

## Parse with explicit formats

`try_to_timestamp` returns null instead of failing when text does not match the stated format. Cast the parsed midnight timestamp to `date` when only a calendar date is required.

In [ ]:
deliveries = deliveries_raw.select(
    'order_id',
    F.expr("try_to_timestamp(order_date_text, 'yyyy-MM-dd')").cast('date').alias('order_date'),
    F.expr("try_to_timestamp(promised_date_text, 'yyyy-MM-dd')").cast('date').alias('promised_date'),
    F.expr("try_to_timestamp(event_time_utc_text, 'yyyy-MM-dd HH:mm:ss')").alias('event_timestamp_utc'),
)

deliveries.show(truncate=False)
deliveries.printSchema()

## Detect invalid values

A null typed value after safe parsing is a data-quality signal. In a production pipeline, retain the raw text or route failed rows to a review table.

In [ ]:
invalid_deliveries = deliveries.filter(
    F.col('order_date').isNull() | F.col('event_timestamp_utc').isNull()
)
invalid_deliveries.show(truncate=False)

## Calendar fields and display labels

Keep typed dates for calculations. Use `date_format` only when a string label is needed for people or an export.

In [ ]:
date_features = deliveries.select(
    'order_id',
    'order_date',
    F.year('order_date').alias('order_year'),
    F.month('order_date').alias('order_month'),
    F.dayofmonth('order_date').alias('day_of_month'),
    F.dayofyear('order_date').alias('day_of_year'),
    F.date_format('order_date', 'dd MMM yyyy').alias('order_date_label'),
)
date_features.show()

## Date and timestamp arithmetic

`datediff` returns whole calendar days. `date_add` adds calendar days, while SQL interval syntax is convenient for timestamp arithmetic.

In [ ]:
delivery_schedule = deliveries.select(
    'order_id',
    'order_date',
    'promised_date',
    F.datediff('promised_date', 'order_date').alias('planned_delivery_days'),
    F.date_add('order_date', 7).alias('follow_up_date'),
    'event_timestamp_utc',
    F.expr('event_timestamp_utc + INTERVAL 2 HOURS').alias('estimated_event_timestamp_utc'),
)
delivery_schedule.show(truncate=False)

## Convert UTC to local time

Use a named time zone rather than manually adding hours. Named zones apply daylight-saving rules.

In [ ]:
deliveries.select(
    'order_id',
    'event_timestamp_utc',
    F.from_utc_timestamp('event_timestamp_utc', 'Europe/Paris').alias('event_timestamp_paris'),
).show(truncate=False)

## Your turn

Create `new_delivery` from `new_delivery_raw`. Parse `ship_date_text` as `ship_date` and `pickup_time_utc_text` as `pickup_timestamp_utc` using the supplied day-first formats. Add `ship_month` and `ship_day_of_year`.

In [ ]:
new_delivery_raw = spark.createDataFrame(
    [('O103', '25/03/2026', '25/03/2026 09:15')],
    ['order_id', 'ship_date_text', 'pickup_time_utc_text'],
)

# Write your solution here.

### Expected result

The row contains ship date `2026-03-25`, ship month 3, and day of year 84. The pickup value has timestamp type.

### Solution - reveal after attempting

In [ ]:
new_delivery = new_delivery_raw.select(
    'order_id',
    F.expr("try_to_timestamp(ship_date_text, 'dd/MM/yyyy')").cast('date').alias('ship_date'),
    F.expr("try_to_timestamp(pickup_time_utc_text, 'dd/MM/yyyy HH:mm')").alias('pickup_timestamp_utc'),
).withColumn(
    'ship_month', F.month('ship_date')
).withColumn(
    'ship_day_of_year', F.dayofyear('ship_date')
)

new_delivery.show(truncate=False)
new_delivery.printSchema()

## Key takeaway

Parse source text explicitly, treat failed parsing as data-quality information, and use named time zones.

**Next:** read from and write to a Fabric Lakehouse.